In [1]:
%load_ext autoreload
%autoreload 2
import matplotlib.pyplot as plt, seaborn as sns
from ss_utils import *
df = load_data(".")   # the folder with train.csv and test.csv

In [2]:
M = "lgbm"
FIXED = dict(n_estimators=400, learning_rate=0.03, num_leaves=15,
             subsample=0.8, subsample_freq=1, colsample_bytree=0.8)

In [3]:
steps = [("G0 raw", ()), ("+G1 cabin", ("G1",)), ("+G2 group", ("G1","G2")),
         ("+G3 spending", ("G1","G2","G3")), ("+G4 age", ("G1","G2","G3","G4"))]
e1, rows = {}, []
for i, (name, grp_set) in enumerate(steps):
    X, y, g, *_ = build(df, impute="logic", groups=grp_set)
    e1[name] = res = run_cv(M, X, y, g, params=FIXED)
    log_result(f"E1-{i}", "E1 cumulative", name, M, res)
    rows.append({"Feature set": name, "# features": X.shape[1],
                 "CV accuracy": f"{res['mean']:.4f} ± {res['std']:.4f}"})
table_e1 = pd.DataFrame(rows); table_e1

lgbm    CV acc = 0.7959 ± 0.0100  (oof 0.7967, 13.7s)
lgbm    CV acc = 0.8079 ± 0.0084  (oof 0.8072, 14.0s)
lgbm    CV acc = 0.8080 ± 0.0090  (oof 0.8098, 13.6s)
lgbm    CV acc = 0.8112 ± 0.0075  (oof 0.8136, 13.8s)
lgbm    CV acc = 0.8127 ± 0.0093  (oof 0.8124, 13.7s)


,Feature set,# features,CV accuracy
0,G0 raw,10,0.7959 ± 0.0100
1,+G1 cabin,14,0.8079 ± 0.0084
2,+G2 group,17,0.8080 ± 0.0090
3,+G3 spending,26,0.8112 ± 0.0075
4,+G4 age,28,0.8127 ± 0.0093


In [7]:
log = pd.read_csv(LOG_PATH)
log.drop_duplicates(subset="run_id", keep="last").to_csv(LOG_PATH, index=False)

In [4]:
names = list(e1)
for a, b in zip(names[:-1], names[1:]):
    paired_test(e1[a], e1[b], a, b)

+G1 cabin - G0 raw: +0.0120 accuracy (better on 15/15 folds), p = 0.000
+G2 group - +G1 cabin: +0.0001 accuracy (better on 8/15 folds), p = 0.926
+G3 spending - +G2 group: +0.0031 accuracy (better on 12/15 folds), p = 0.032
+G4 age - +G3 spending: +0.0015 accuracy (better on 9/15 folds), p = 0.127


In [5]:
full, rows = ("G1","G2","G3","G4"), []
for drop in full:
    X, y, g, *_ = build(df, impute="logic", groups=tuple(x for x in full if x != drop))
    res = run_cv(M, X, y, g, params=FIXED)
    log_result(f"E1-wo{drop}", "E1 leave-one-out", f"all minus {drop}", M, res)
    rows.append({"Removed": drop, "CV accuracy": round(res["mean"], 4),
                 "Change vs full": round(res["mean"] - e1["+G4 age"]["mean"], 4)})
table_e1b = pd.DataFrame(rows); table_e1b

lgbm    CV acc = 0.7952 ± 0.0084  (oof 0.7978, 13.5s)
lgbm    CV acc = 0.8111 ± 0.0090  (oof 0.8100, 13.8s)
lgbm    CV acc = 0.8079 ± 0.0090  (oof 0.8082, 13.7s)
lgbm    CV acc = 0.8112 ± 0.0075  (oof 0.8136, 13.6s)


,Removed,CV accuracy,Change vs full
0,G1,0.7952,-0.0175
1,G2,0.8111,-0.0016
2,G3,0.8079,-0.0048
3,G4,0.8112,-0.0015


In [6]:
IMPUTERS["logic_noR2b"] = lambda d: impute_logic(d, cryo_from_zero_spend=False)
e2, rows = {}, []
for model in ["lgbm", "logreg"]:
    for imp in ["native", "simple", "logic", "logic_noR2b"]:
        if model == "logreg" and imp == "native":
            continue                      # logreg cannot take NaN
        X, y, g, *_ = build(df, impute=imp)
        e2[(model, imp)] = res = run_cv(model, X, y, g, params=FIXED if model == "lgbm" else None)
        log_result(f"E2-{model}-{imp}", "E2 imputation", imp, model, res)
        rows.append({"Model": model, "Imputation": imp,
                     "CV accuracy": f"{res['mean']:.4f} ± {res['std']:.4f}"})
table_e2 = pd.DataFrame(rows); display(table_e2)
paired_test(e2[("lgbm","simple")], e2[("lgbm","logic")], "simple", "logic")
paired_test(e2[("logreg","simple")], e2[("logreg","logic")], "simple (logreg)", "logic (logreg)")

lgbm    CV acc = 0.8104 ± 0.0073  (oof 0.8120, 14.0s)
lgbm    CV acc = 0.8114 ± 0.0090  (oof 0.8131, 13.7s)
lgbm    CV acc = 0.8127 ± 0.0093  (oof 0.8124, 13.7s)
lgbm    CV acc = 0.8114 ± 0.0090  (oof 0.8119, 13.7s)
logreg  CV acc = 0.7978 ± 0.0082  (oof 0.7974, 1.6s)
logreg  CV acc = 0.7980 ± 0.0092  (oof 0.7980, 1.5s)
logreg  CV acc = 0.7980 ± 0.0083  (oof 0.7977, 1.5s)


,Model,Imputation,CV accuracy
0,lgbm,native,0.8104 ± 0.0073
1,lgbm,simple,0.8114 ± 0.0090
2,lgbm,logic,0.8127 ± 0.0093
3,lgbm,logic_noR2b,0.8114 ± 0.0090
4,logreg,simple,0.7978 ± 0.0082
5,logreg,logic,0.7980 ± 0.0092
6,logreg,logic_noR2b,0.7980 ± 0.0083


logic - simple: +0.0013 accuracy (better on 10/15 folds), p = 0.292
logic (logreg) - simple (logreg): +0.0002 accuracy (better on 8/15 folds), p = 0.792


(0.00015371935909795907, 0.7917242622917567)

In [8]:
*_, counts = build(df, impute="logic")
pd.Series(counts, name="values filled").to_frame()

,values filled
R1 cryo -> spend 0,531
R2a spend>0 -> awake,174
R2b zero spend -> asleep,123
R3 child -> spend 0,67
R4a planet <- group,131
R4b planet <- surname,144
R4c planet <- deck,13
R5 deck <- group,137
R5 side <- group,137
R5 surname <- group,139


In [9]:
for name, t in {"e1_cumulative": table_e1, "e1_leave_one_out": table_e1b, "e2_imputation": table_e2}.items():
    t.to_csv(f"results/{name}.csv", index=False)
    t.to_latex(f"results/{name}.tex", index=False)